## Troubleshooting — control plane & worker nodes

### Control plane

**`kubectl` says "connection refused" / "i/o timeout"** — the API server is unreachable. From the control-plane node:

```bash
sudo crictl ps | grep apiserver     # is the static pod running?
sudo journalctl -u kubelet -n 50    # kubelet logs say why a static pod failed
ls /etc/kubernetes/manifests/       # manifests still there?
```

Causes: a **bad flag** you edited into `kube-apiserver.yaml` (kubelet logs are explicit), an **expired cert** (`kubeadm certs check-expiration`), or **etcd down** (the API server won't start without it).

**Pods stuck `ContainerCreating` cluster-wide** — usually a **broken CNI** (`kubectl get pods -n kube-system | grep -E 'calico|cilium|flannel'`) or the kubelet can't reach the API server.

**Scheduler / controller-manager idle** — `kubectl logs -n kube-system kube-scheduler-<node>`; look for a lost **leader-election lease** or missing RBAC.

### Worker nodes

**Node `NotReady`** — read `kubectl describe node <name>` Conditions (`Ready`, `MemoryPressure`, `DiskPressure`, `PIDPressure`, `NetworkUnavailable`), then SSH:

```bash
sudo systemctl status kubelet          # running?
sudo journalctl -u kubelet -n 200      # read backwards to the first complaint
sudo systemctl status containerd       # runtime up?
df -h                                  # full disk → NotReady
```

`crictl ps` speaks CRI directly when the kubelet is down. Quick fixes: `kubectl cordon`/`drain --ignore-daemonsets` for maintenance, `uncordon` when fixed, `systemctl restart kubelet`. On our map this section tours the **Control Plane** and **Node runtime** boxes — the two places a whole-cluster outage originates.